# Classificação de Retinopatia Hipertensiva em Imagens de Fundo de Olho

> **Projeto de Pesquisa Científica**

Pipeline de Deep Learning para detecção automática de **retinopatia hipertensiva** em imagens de fundoscopia ocular.

Caso queira entender a lógica por trás, aqui está o repositório utilizado no projeto: https://github.com/Bappoz/fundus-classification.git

| Atributo | Descrição |
|---|---|
| **Tarefa** | Classificação binária: Normal vs. Retinopatia Hipertensiva |
| **Abordagem** | Transfer learning com backbone CNN pré-treinado em ImageNet |
| **Métrica principal** | AUC-ROC (dado desbalanceamento de classes) |
| **Plataforma** | Google Colab (GPU T4 / A100) |

---

**Sumário**
1. [Configuração do Ambiente](#1-configuracao)
2. [Verificação de Hardware](#2-hardware)
3. [Análise Exploratória dos Dados (EDA)](#3-eda)
   - 3.1 Inventário do Dataset
   - 3.2 Estrutura por Paciente
   - 3.3 Distribuição de Classes e Fontes
   - 3.4 Amostras Visuais

---
## 1. Configuração do Ambiente

Detecta automaticamente o ambiente (Colab vs. local), monta o Google Drive, clona/atualiza o repositório e instala as dependências necessárias.

### Dataset Utilizado

Este projeto usa dois datasets públicos de fundoscopia ocular, combinados e filtrados:

| Dataset | Fonte | Uso |
|---|---|---|
| **BRSET** | Brazilian Multilabel Ophthalmological Dataset (USP) | Normal + Retinopatia Hipertensiva |
| **ODIR-5K** | Ocular Disease Intelligent Recognition (ODIR Challenge) | Retinopatia Hipertensiva adicional |

#### Como obter o dataset

**Opção A — Colaborador do projeto**
> Solicite o arquivo `dataset.zip` ao orientador e salve-o em `Meu Drive/`.

**Opção B — Configurar do zero com os dados públicos**
1. Baixe o .zip (meta e images(odir5k e brset)): https://drive.google.com/file/d/1LL-8X1uCwgXRu0F_u2Q8mNDxEM2ylPcn/view?usp=sharing 
2. Monte a estrutura abaixo, compacte como `dataset.zip` e faça upload para `Meu Drive/`

#### Estrutura interna esperada do `dataset.zip`

O zip deve conter uma pasta `retino/` na raiz:

```
dataset.zip
└── retino/
    ├── meta/
    │   ├── labels_brset_filt.xlsx
    │   └── data_filt.xlsx
    └── images/
        ├── normal/
        ├── hr_brset/
        └── hr_odir5k/
```

Na **primeira execução** o zip é extraído para o Drive. Nas execuções seguintes a extração é pulada automaticamente.

#### ⚠ Nome do arquivo diferente?

Edite a variável `DRIVE_ZIP` no início da próxima célula:

```python
DRIVE_ZIP = "/content/drive/MyDrive/dataset.zip"  # ← altere aqui se necessário
```

In [ ]:
import os, sys

# ─── CONFIG ───────────────────────────────────────────────────────────────────
DRIVE_ZIP  = "/content/drive/MyDrive/dataset.zip"   # zip no seu Drive
DRIVE_ROOT = "/content/drive/MyDrive/dataset"         # destino da extração (não alterar)
REPO       = "https://github.com/Bappoz/fundus-classification.git"
# ──────────────────────────────────────────────────────────────────────────────

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    print("Ambiente local detectado — pulando mount do Drive.")
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive")

    meta_ok   = os.path.isdir(f"{DRIVE_ROOT}/labels_brset_filt.xlsx")
    images_ok = os.path.isdir(f"{DRIVE_ROOT}/normal")

    if meta_ok and images_ok:
        print(f"✓ Dataset já extraído em {DRIVE_ROOT} — pulando unzip.")
    elif os.path.isfile(DRIVE_ZIP):
        print(f"  Extraindo {DRIVE_ZIP} para o Drive (apenas na primeira vez)...")
        !unzip -q -o "{DRIVE_ZIP}" -d "/content/drive/MyDrive/"
        print(f"✓ Dataset extraído em {DRIVE_ROOT}.")
    else:
        print(f"⚠ Arquivo não encontrado: {DRIVE_ZIP}")
        print("  Verifique a variável DRIVE_ZIP ou consulte a célula de documentação acima.")

    # Repositório
    %cd /content
    if not os.path.isdir("/content/fundus-classification"):
        !git clone {REPO}
    else:
        print("✓ Repositório já clonado — atualizando...")
        !cd /content/fundus-classification && git pull
    %cd /content/fundus-classification
    sys.path.insert(0, os.path.abspath("src"))
else:
    src_path = os.path.abspath(os.path.join(os.getcwd(), "..", "src"))
    if src_path not in sys.path:
        sys.path.insert(0, src_path)

if IN_COLAB:
    %pip install -q -e .
else:
    %pip install -q -e ..

%pip install -q timm albumentations
print("✓ Ambiente configurado com sucesso.")

---
## 2. Verificação de Hardware

Confirma disponibilidade de GPU e carrega a configuração do experimento.

> **Sem GPU?** Acesse *Runtime → Change runtime type → GPU (T4)* antes de prosseguir.

In [ ]:
import torch
from retino.config import Config as cfg

gpu_ok      = torch.cuda.is_available()
device_name = torch.cuda.get_device_name(0) if gpu_ok else "N/A"

print(f"{'✓' if gpu_ok else '✗'} GPU       : {device_name}")
print(f"  Device    : {cfg.device}")
print(f"  Backbone  : {cfg.backbone}")

if not gpu_ok:
    raise RuntimeError("GPU não disponível — altere o runtime antes de prosseguir.")

---
## 3. Análise Exploratória dos Dados (EDA)

### 3.1 Inventário do Dataset

Carrega os rótulos, verifica a integridade dos arquivos e inspeciona a distribuição de classes e fontes de dados.

In [ ]:
from retino.data.loader import build_labels, verify_files
from retino.config import cfg
from pathlib import Path

# ajuste para o caminho real onde está o dataset.zip descompactado
cfg.data_root  = Path("/content/drive/MyDrive")   # pasta raiz no Drive
cfg.meta_dir   = Path("/content/drive/MyDrive/dataset")   # onde estão os .xlsx
cfg.image_dir  = Path("/content/drive/MyDrive/dataset")   # onde estão normal/ hr_brset/ hr_odir5k/

df = build_labels()
df = verify_files(df)

n_total  = len(df)
n_normal = (df.label == 0).sum()
n_hiper  = (df.label == 1).sum()
ratio    = n_normal / n_hiper

print(f"Total de imagens  : {n_total:,}")
print(f"  Normal          : {n_normal:,}  ({100 * n_normal / n_total:.1f}%)")
print(f"  Hipertensiva    : {n_hiper:,}  ({100 * n_hiper / n_total:.1f}%)")
print(f"  Ratio neg:pos   : {ratio:.1f}:1")
print()
print("Distribuição por fonte:")
src_table = (
    df.groupby(["source", "label"])
      .size()
      .rename("n")
      .rename({0: "normal", 1: "hipertensiva"}, level="label")
)
print(src_table.to_string())

### 3.2 Estrutura por Paciente

O split treino/validação deve ser feito **por paciente** (e não por imagem) para evitar *data leakage* — imagens do mesmo paciente podem aparecer em treino e teste caso o split seja feito por imagem.

In [ ]:
imgs_por_paciente = df.groupby("patient_id").size()

print(f"Pacientes únicos       : {df.patient_id.nunique():,}")
print(f"Imagens / paciente     : média={imgs_por_paciente.mean():.2f}  máx={imgs_por_paciente.max()}")
print(f"Pacientes com > 1 img  : {(imgs_por_paciente > 1).sum():,}")
print()
print("Split DEVE ser por patient_id para evitar data leakage.")

### 3.3 Distribuição de Classes e Fontes

Visualização do desbalanceamento de classes e contribuição de cada fonte de dados.

In [ ]:
import matplotlib.pyplot as plt

label_names = {0: "Normal", 1: "Hipertensiva"}
palette     = {"Normal": "#4C9BE8", "Hipertensiva": "#E85C4C"}

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
fig.suptitle("Distribuição do Dataset", fontsize=14, fontweight="bold")

# Contagem total por classe
counts = df["label"].map(label_names).value_counts()
bars = axes[0].bar(
    counts.index, counts.values,
    color=[palette[k] for k in counts.index],
    edgecolor="white", linewidth=0.8, width=0.5
)
for bar, val in zip(bars, counts.values):
    axes[0].text(
        bar.get_x() + bar.get_width() / 2, bar.get_height() + max(counts.values) * 0.02,
        f"{val:,}", ha="center", va="bottom", fontsize=11, fontweight="bold"
    )
axes[0].set_title("Contagem por Classe", pad=10)
axes[0].set_ylabel("Num. de Imagens")
axes[0].set_ylim(0, max(counts.values) * 1.15)
axes[0].spines[["top", "right"]].set_visible(False)

# Contagem por fonte e classe
src_pivot = (
    df.assign(classe=df["label"].map(label_names))
      .groupby(["source", "classe"])
      .size()
      .unstack(fill_value=0)
)
src_pivot.plot(
    kind="bar", ax=axes[1],
    color=[palette[c] for c in src_pivot.columns],
    edgecolor="white", linewidth=0.8, rot=30
)
axes[1].set_title("Contagem por Fonte e Classe", pad=10)
axes[1].set_xlabel("Fonte", labelpad=8)
axes[1].set_ylabel("Num. de Imagens")
axes[1].spines[["top", "right"]].set_visible(False)
axes[1].legend(title="Classe", framealpha=0.8)

plt.tight_layout()
plt.savefig("eda_distribuicao.png", dpi=150, bbox_inches="tight")
plt.show()

### 3.4 Amostras Visuais

Grade comparativa de imagens de fundo de olho para as duas classes, permitindo inspeção qualitativa das características visuais.

In [ ]:
import cv2
import matplotlib.pyplot as plt

def show_samples(df, n=4, seed=42):
    label_names = {0: "Normal", 1: "Hipertensiva"}
    row_colors  = {0: "#4C9BE8", 1: "#E85C4C"}

    fig, axes = plt.subplots(2, n, figsize=(3 * n, 7))
    fig.suptitle(
        "Amostras: Normal vs Retinopatia Hipertensiva",
        fontsize=13, fontweight="bold"
    )

    for row, lab in enumerate([0, 1]):
        subset = df[df.label == lab].sample(
            min(n, (df.label == lab).sum()), random_state=seed
        )
        for col, (_, r) in enumerate(subset.iterrows()):
            img = cv2.cvtColor(cv2.imread(str(r.path)), cv2.COLOR_BGR2RGB)
            axes[row, col].imshow(img)
            axes[row, col].axis("off")
            axes[row, col].set_title(r.source, fontsize=8, color="#555555")
        axes[row, 0].set_ylabel(
            label_names[lab], fontsize=12,
            fontweight="bold", color=row_colors[lab],
            rotation=90, labelpad=12
        )

    plt.tight_layout()
    plt.savefig("eda_amostras.png", dpi=150, bbox_inches="tight")
    plt.show()

show_samples(df)